# ⚖️ 롱숏 중립화 팩터 모델 (Market-Neutral Long-Short)

## 왜 롱숏인가
지금까지의 "상위 20% 롱"은 수익의 상당 부분이 **시장 베타**(시장이 오르면 다 오름)다.
**상위 롱 − 하위 숏**(달러 중립)을 하면 시장 방향이 상쇄되어, **팩터(상대적 순위) 자체의 순수 알파**만 남는다.

> **핵심 판정: 롱숏이 시장 베타 ≈ 0 인데 Sharpe > 0 이면 → 진짜 팩터 알파.**
> (롱온리 Sharpe가 높아도 그건 대부분 "시장이 올라서"일 뿐)

## 구성
- 매 리밸런스: 복합 팩터점수 **상위 20% 롱 + 하위 20% 숏** (각 동일비중, 달러 중립)
- 가격 3팩터(모멘텀·저변동성·반전) — SEC 없이 바로 실행 가능
- 거래비용 **양쪽 다리 모두** 차감 · 시장(SPY) 베타 회귀로 중립성 확인

## 정직한 예상
대형주는 팩터가 상당히 arbitraged 되어, **롱숏 알파가 작거나(≈0) 비용 후 마이너스**일 수 있다.
그게 나오면 → "대형주에선 팩터 엣지가 약하다"는 정직한 증거(소형주·해외로 가야 함).

> ⚠️ 숏은 차입비용·업틱룰 등 현실 마찰이 큼. 여기선 팩터 검증용 분석이며 실거래 권장 아님.

In [ ]:
# =====================================================================
# ⚙️ Cell 1: 설정 & 데이터
# =====================================================================
import numpy as np, pandas as pd, yfinance as yf
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
try: plt.rcParams['font.family']='Malgun Gothic'
except Exception: pass
plt.rcParams['axes.unicode_minus']=False

UNIVERSE=['AAPL','MSFT','GOOGL','AMZN','META','NVDA','AVGO','ORCL','CRM','ADBE','AMD','QCOM','TXN','INTC','CSCO',
 'JPM','BAC','WFC','GS','MS','C','V','MA','AXP','BLK','UNH','JNJ','LLY','PFE','ABBV','MRK','TMO','ABT',
 'PG','KO','PEP','WMT','COST','HD','MCD','NKE','SBUX','DIS','XOM','CVX','CAT','BA','HON','GE','UPS','NFLX','T','VZ']
BENCH='SPY'; START='2012-01-01'; END=None; COST=0.0005; ANN=252; REBAL=21; Q=0.20

def fetch_close(tickers):
    fr={}
    for t in tickers:
        try:
            df=yf.download(t,start=START,end=END,auto_adjust=True,progress=False)
            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(0)
            s=df['Close'].dropna()
            if len(s)>300: fr[t]=s
        except Exception: pass
    return pd.DataFrame(fr)

print('수집 중...')
allpx=fetch_close(UNIVERSE+[BENCH])
spy=allpx[BENCH]
close=allpx[[c for c in allpx.columns if c!=BENCH]].dropna(axis=1,thresh=int(len(allpx)*0.8)).dropna()
spy=spy.reindex(close.index).ffill()
print(f'{close.shape[1]}종목 × {close.shape[0]}일')

In [ ]:
# =====================================================================
# 🧮 Cell 2: 가격 팩터 + 복합 점수
# =====================================================================
rets=close.pct_change()
f_mom=close.shift(21)/close.shift(252)-1
f_lowvol=-(rets.rolling(126).std())
f_rev=-(close.pct_change(21))
FACTORS={'Momentum':f_mom,'LowVol':f_lowvol,'Reversal':f_rev}
def zrow(r):
    m,s=r.mean(),r.std(); return (r-m)/s if (s and s>0) else r*0.0
def composite(d):
    return sum(zrow(f.loc[d]) for f in FACTORS.values())/len(FACTORS)
print('팩터 준비 완료')

In [ ]:
# =====================================================================
# ⚖️ Cell 3: 롱숏 백테스트 (상위 롱 − 하위 숏, 달러중립, 비용 양다리)
# =====================================================================
dates=close.index; ridx=list(range(252,len(dates)-1,REBAL))
def perf(pr):
    r=np.asarray(pr,float); out={'CAGR':np.nan,'Sharpe':0.0,'MDD':0.0}
    if len(r)==0: return out
    ppy=ANN/REBAL; eq=np.cumprod(1+r); pk=np.maximum.accumulate(eq)
    out['CAGR']=float(eq[-1]**(ppy/len(r))-1) if eq[-1]>0 else np.nan
    out['Sharpe']=float(r.mean()/r.std()*np.sqrt(ppy)) if r.std()>0 else 0.0
    out['MDD']=float(((eq-pk)/pk).min()); return out

ls,lo,mk,dd=[],[],[],[]
prev_l,prev_s=set(),set()
for i in ridx:
    d=dates[i]; sc=composite(d).dropna()
    if len(sc)<10: continue
    k=max(int(len(sc)*Q),1)
    srt=sc.sort_values(ascending=False)
    longs=list(srt.head(k).index); shorts=list(srt.tail(k).index)
    j=min(i+REBAL,len(dates)-1); d2=dates[j]
    lr=float((close.loc[d2,longs]/close.loc[d,longs]-1).mean())
    sr=float((close.loc[d2,shorts]/close.loc[d,shorts]-1).mean())
    nl=set(longs); ns=set(shorts)
    tcost=((len(nl-prev_l)+len(prev_l-nl))+(len(ns-prev_s)+len(prev_s-ns)))/max(k,1)*COST
    ls.append((lr-sr)-tcost)                 # 롱숏(달러중립)
    lo.append(lr-((len(nl-prev_l)+len(prev_l-nl))/max(k,1))*COST)   # 롱온리(참고)
    mk.append(float(spy.loc[d2]/spy.loc[d]-1))                       # 시장(SPY)
    dd.append(d2); prev_l,prev_s=nl,ns

ls=pd.Series(ls,index=dd); lo=pd.Series(lo,index=dd); mk=pd.Series(mk,index=dd)
mL,mLO,mM=perf(ls.values),perf(lo.values),perf(mk.values)
# 롱숏의 시장 베타 (0에 가까울수록 중립)
beta=float(np.cov(ls.values,mk.values)[0,1]/np.var(mk.values)) if np.var(mk.values)>0 else np.nan

print('=== 성과 비교 (월간 리밸런스, 비용반영) ===')
print(f'{"":16}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}')
print(f'{"롱숏(중립)":16}{mL["CAGR"]*100:8.1f}%{mL["Sharpe"]:9.2f}{mL["MDD"]*100:8.1f}%')
print(f'{"롱온리(상위20%)":16}{mLO["CAGR"]*100:8.1f}%{mLO["Sharpe"]:9.2f}{mLO["MDD"]*100:8.1f}%')
print(f'{"시장(SPY)":16}{mM["CAGR"]*100:8.1f}%{mM["Sharpe"]:9.2f}{mM["MDD"]*100:8.1f}%')
print(f'\n📐 롱숏의 시장 베타: {beta:+.3f}  (0에 가까우면 시장중립 성공)')
print('판정: 롱숏 Sharpe>0 & 베타≈0 → 순수 팩터 알파 존재. 아니면 → 대형주 팩터 엣지 약함.')

In [ ]:
# =====================================================================
# 📐 Cell 3.5: 순수 팩터 알파 측정 (시장 회귀 → 절편=알파, t-stat)
#   베타가 0이 아니면 Sharpe만으론 오해. 시장성분을 제거한 '알파'가 진짜 팩터 실력.
# =====================================================================
x = mk.values; y = ls.values
beta_reg  = float(np.cov(y, x)[0, 1] / np.var(x)) if np.var(x) > 0 else 0.0
alpha_per = float(y.mean() - beta_reg * x.mean())          # 리밸런스 기간당 알파
ppy       = ANN / REBAL
alpha_ann = alpha_per * ppy                                 # 연율화 알파
resid     = y - (alpha_per + beta_reg * x)
se        = resid.std(ddof=2) / np.sqrt(len(y)) if len(y) > 2 else np.nan
t_alpha   = alpha_per / se if (se and se > 0) else 0.0
ls_hedged = pd.Series(y - beta_reg * x, index=ls.index)     # 베타 헤지 = 순수 알파 스트림
mh = perf(ls_hedged.values)

print('=== 롱숏 순수 알파 (시장 회귀) ===')
print(f'  베타            : {beta_reg:+.3f}')
print(f'  알파(연율화)    : {alpha_ann*100:+.2f}%')
print(f'  알파 t-stat     : {t_alpha:+.2f}   (|t|>2 면 통계적으로 유의)')
print(f'  베타헤지 롱숏 Sharpe : {mh["Sharpe"]:.2f}   (시장 성분 제거한 순수 팩터 수익)')
print()
if abs(t_alpha) > 2 and alpha_ann > 0:
    print('  → ✅ 알파 유의(+). 대형주에서도 팩터 엣지 존재 신호.')
elif alpha_ann <= 0:
    print('  → ⚠️ 알파 0 이하. 대형주 팩터 엣지 없음(정직한 결과) → 소형주/다른 팩터로.')
else:
    print('  → 🤔 알파는 +지만 t값 약함 = 노이즈일 수 있음. 유의성 부족.')


In [ ]:
# =====================================================================
# 📈 Cell 4: 자산곡선 & 낙폭
# =====================================================================
fig,ax=plt.subplots(2,1,figsize=(13,9))
(1+ls).cumprod().plot(ax=ax[0],label='롱숏(시장중립)',lw=2.4,color='purple')
(1+lo).cumprod().plot(ax=ax[0],label='롱온리 상위20%',lw=1.4,color='navy')
(1+mk).cumprod().plot(ax=ax[0],label='시장(SPY)',lw=1.4,ls='--',color='gray')
ax[0].set_yscale('log'); ax[0].set_title('자산곡선(로그) — 롱숏은 시장과 무관하게 움직여야 정상')
ax[0].legend(); ax[0].grid(alpha=0.3)
for nm,sr in [('롱숏',ls),('롱온리',lo),('시장',mk)]:
    eq=(1+sr).cumprod(); (eq/eq.cummax()-1).mul(100).plot(ax=ax[1],label=nm,lw=1.4)
ax[1].set_title('낙폭(Drawdown, %)'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 🧾 해석

| 지표 | 의미 |
|---|---|
| **롱숏 Sharpe** | 시장 방향을 제거한 **순수 팩터 알파**. >0이면 팩터가 실제로 순위를 맞힘. |
| **롱숏 베타** | 0에 가까울수록 시장중립 성공(시장이 올라도/내려도 무관). |
| **롱온리 vs 시장** | 롱온리가 좋아 보여도 대부분 시장 베타. 롱숏이 진짜 실력을 드러냄. |
| **롱숏 MDD** | 시장 폭락과 무관해야 함. 롱온리보다 낙폭이 다른 시점에 오면 분산 가치. |

### 정직한 결론 가능성
- **롱숏 Sharpe ≈ 0 또는 음수(비용 후)** → 대형주에선 팩터 엣지가 약하다는 증거. 정상이고 정직한 결과.
- **롱숏 Sharpe 뚜렷이 >0 & 베타≈0** → 순수 팩터 알파 존재. 다만 숏 마찰(차입비용) 감안 필요.

### 다음 장
1. **소형주/중형주 유니버스** — 팩터 프리미엄이 대형주보다 강함 (엣지 있는 곳으로)
2. **PIT 밸류·퀄리티 결합** (`US_factor_pit.ipynb`의 5팩터를 롱숏에 적용)
3. **섹터 중립화** — 섹터 편중 제거 후 순수 종목선택 알파
4. **숏 마찰 모델링** — 차입비용·유동성 반영한 현실적 롱숏

> ⚠️ 파라미터 고정(관습값). 숏은 현실 마찰 큼 — 검증용 분석이지 실거래 권장 아님.